# KaraokeForge Worker — Colab Notebook

Chạy pipeline karaoke (tách nhạc, nhận diện lời, render video) trên Google Colab GPU.

**Yêu cầu:**
- Runtime GPU (khuyến nghị T4 trở lên): `Runtime > Change runtime type > GPU`.
- 1 tài khoản Google Drive duy nhất (v1 chạy single-account — chứa dữ liệu job, model cache, output).
- Chạy lần lượt các cell theo thứ tự 1 → 9. Cell 8 (Test Pipeline) là tuỳ chọn.

**Cảnh báo:** Colab free tier ngắt session sau khoảng 12 giờ (hoặc sớm hơn nếu rảnh quá
lâu). Worker lưu checkpoint sau mỗi stage (`checkpoints.*` trong job JSON) — session bị
ngắt giữa chừng thì lần chạy notebook kế tiếp sẽ tự resume từ checkpoint cuối cùng,
không chạy lại từ đầu (`outputs/{job_id}/` trên Drive giữ dữ liệu qua các phiên).

| Cell | Vai trò |
|---|---|
| 2 | Config (`WORKER_ID`, `DRIVE_ROOT`, `REPO_URL`...) |
| 3 | Cài đặt (ffmpeg, font, clone repo, `pip install`) |
| 4 | Mount Google Drive + tạo cấu trúc thư mục |
| 5 | Kiểm tra GPU + auto-select model |
| 6 | Warm cache Demucs |
| 7 | Import code pipeline |
| 8 | (Tuỳ chọn) Test nhanh pipeline |
| 9 | Chạy worker (`run_forever()`) |
| 10 | Thao tác thủ công (recover stale, thống kê, retry) |

In [ ]:
# Cell 2 — Config
WORKER_ID = "worker_colab_1"        # đổi tên nếu chạy nhiều worker song song
DRIVE_ROOT = "/content/drive/MyDrive/KaraokeForge"   # path cố định (contract D5)
REPO_URL = "https://github.com/Minnyat/Karaoke-hub-worker"   # repo PUBLIC (worker) — Colab clone không cần token

POLL_INTERVAL = 15       # giây giữa các lần poll khi hàng đợi rỗng
STALE_AFTER_MIN = 10     # heartbeat cũ hơn ngưỡng này (phút) -> job coi là stale (contract D3.5)

print(f"WORKER_ID   = {WORKER_ID}")
print(f"DRIVE_ROOT  = {DRIVE_ROOT}")
print(f"REPO_URL    = {REPO_URL}")

In [ ]:
# Cell 3 — Cài đặt (ffmpeg, font, clone repo, Python deps)
!apt-get -qq install -y ffmpeg

# Font Be Vietnam Pro (giấy phép OFL) tải trực tiếp từ github.com/google/fonts.
# KHÔNG dùng URL font không tin cậy của bản PRD cũ (xem contract §6.5).
!mkdir -p /usr/share/fonts/custom
!wget -q -O /usr/share/fonts/custom/BeVietnamPro-Bold.ttf \
    https://raw.githubusercontent.com/google/fonts/main/ofl/bevietnampro/BeVietnamPro-Bold.ttf
!wget -q -O /usr/share/fonts/custom/BeVietnamPro-Regular.ttf \
    https://raw.githubusercontent.com/google/fonts/main/ofl/bevietnampro/BeVietnamPro-Regular.ttf
!fc-cache -f

!git clone --depth 1 $REPO_URL /content/karaokeforge 2>/dev/null || \
    (cd /content/karaokeforge && git pull)

!pip install -q -r /content/karaokeforge/worker/requirements.txt
print("Cài đặt xong.")

In [ ]:
# Cell 4 — Mount Google Drive + tạo cấu trúc thư mục (contract §2)
import os

from google.colab import drive

drive.mount("/content/drive")

FOLDERS = [
    f"{DRIVE_ROOT}/queue/pending",
    f"{DRIVE_ROOT}/queue/processing",
    f"{DRIVE_ROOT}/queue/completed",
    f"{DRIVE_ROOT}/queue/failed",
    f"{DRIVE_ROOT}/uploads",
    f"{DRIVE_ROOT}/outputs",
    f"{DRIVE_ROOT}/models_cache",
    "/content/temp",   # Config.TEMP_DIR — local SSD, KHÔNG nằm trên Drive
]
for path in FOLDERS:
    os.makedirs(path, exist_ok=True)
    print(f"OK  {path}")

In [ ]:
# Cell 5 — Kiểm tra GPU + auto-select model theo VRAM
import sys

sys.path.insert(0, "/content/karaokeforge/worker")

!nvidia-smi

from karaokeforge.config import Config
from karaokeforge.utils.gpu import detect_gpu

gpu_info = detect_gpu()
print("GPU:", gpu_info)

Config.auto_select_models()
print("Demucs model:       ", Config.DEFAULT_DEMUCS_MODEL)
print("Whisper model:       ", Config.DEFAULT_WHISPER_MODEL)
print("Whisper compute type:", Config.WHISPER_COMPUTE_TYPE)

In [ ]:
# Cell 6 — Warm cache Demucs (tải + cache model trước khi vào vòng xử lý job)
# Whisper KHÔNG được load ở đây — load lazy trong stage 2, tránh giữ 2 model
# cùng lúc trên GPU (CLAUDE.md #3: Demucs -> unload -> Whisper -> unload).
from karaokeforge.pipeline.separator import AudioSeparator

_separator = AudioSeparator()
_separator.load_model(Config.DEFAULT_DEMUCS_MODEL)
_separator.unload()
print("Đã cache model Demucs:", Config.DEFAULT_DEMUCS_MODEL)

In [ ]:
# Cell 7 — Import code pipeline (đã clone ở Cell 3, KHÔNG clone lại ở đây)
import os
import sys

sys.path.insert(0, "/content/karaokeforge/worker")

from karaokeforge.config import Config
from karaokeforge.drive import DriveQueue, DriveStorage
from karaokeforge.worker import KaraokeWorker

font_path = f"{Config.FONT_DIR}/{Config.DEFAULT_FONT}"
assert os.path.exists(font_path), f"Thiếu font {font_path} — kiểm tra lại Cell 3"
print("Import OK. Font sẵn sàng:", font_path)

In [ ]:
# Cell 8 — (Tuỳ chọn) Test nhanh pipeline trên 1 job có sẵn trong queue/pending/
# Đặt RUN_SMOKE_TEST = True nếu muốn chạy process_job() thủ công cho đúng 1 job
# (kiểm tra font tiếng Việt / cấu hình) trước khi chạy vòng lặp bất tận ở Cell 9.
RUN_SMOKE_TEST = False

if RUN_SMOKE_TEST:
    queue = DriveQueue(DRIVE_ROOT)
    job = queue.poll_and_claim(WORKER_ID)
    if job is None:
        print("Không có job nào trong queue/pending/ để test.")
    else:
        worker = KaraokeWorker(WORKER_ID, DRIVE_ROOT, queue=queue)
        worker.process_job(job)
        queue.mark_completed(job)
        print("Job test hoàn tất:", job["id"])
else:
    print("Bỏ qua smoke test (RUN_SMOKE_TEST=False).")

In [ ]:
# Cell 9 — Chạy worker (vòng lặp bất tận: poll -> process -> mark_completed/failed)
# Giữ tab Colab mở để tránh session bị thu hồi (PRD §3.4 keep-alive).
# Dừng: bấm nút Stop (■) trên cell này (KeyboardInterrupt) — job đang dở (nếu có)
# ở lại queue/processing/, worker lần sau (hoặc recover_stale_jobs ở Cell 10) sẽ resume.
worker = KaraokeWorker(WORKER_ID, DRIVE_ROOT)
worker.run_forever()

In [ ]:
# Cell 10 — Thao tác thủ công
import json
import shutil

from karaokeforge.drive import DriveQueue

manual_queue = DriveQueue(DRIVE_ROOT)

# Đưa job stale (heartbeat cũ hơn STALE_AFTER_MIN phút) về pending/ để worker khác nhận
# — recover_stale_jobs() KHÔNG nhận tham số (drive/queue.py).
recovered = manual_queue.recover_stale_jobs()
print("Đã recover:", recovered)

# Thống kê nhanh 4 folder (DriveQueue không có print_stats()/retry_failed_job() —
# viết inline bằng os/json thay vì thêm method vào drive/queue.py).
for name in ("pending", "processing", "completed", "failed"):
    folder = f"{DRIVE_ROOT}/queue/{name}"
    count = len([f for f in os.listdir(folder) if f.endswith(".json")])
    print(f"{name}: {count} job")

# Retry 1 job failed cụ thể — điền FAILED_JOB_ID rồi chạy lại cell này.
FAILED_JOB_ID = None  # ví dụ: "job_a1b2c3d4"
if FAILED_JOB_ID:
    src = f"{DRIVE_ROOT}/queue/failed/{FAILED_JOB_ID}.json"
    dst = f"{DRIVE_ROOT}/queue/pending/{FAILED_JOB_ID}.json"
    with open(src, "r", encoding="utf-8") as f:
        job = json.load(f)
    job["status"] = "pending"
    job["attempts"] = 0
    job["error"] = None
    with open(src, "w", encoding="utf-8") as f:
        json.dump(job, f, indent=2, ensure_ascii=False)
    shutil.move(src, dst)
    print("Đã đưa job", FAILED_JOB_ID, "về pending/ để thử lại.")